# DTM Generation Pipeline — From Trained Model to Drainage-Ready GeoTIFF

## Purpose
This notebook takes the trained ground-classification model (from the training notebook)
and produces a clean, hydrologically conditioned Digital Terrain Model (DTM)
ready for drainage network design.

## Full pipeline
```
Trained model (.pth)
        ↓
Step 1: Inference on ALL tiles (full resolution, not sub-sampled)
        ↓
Step 2: Collect ONLY ground points (class=1)
        ↓
Step 3: IDW interpolation → continuous raster (removes vegetation, buildings)
        ↓
Step 4: Hydrological conditioning (Priority-Flood sink-filling)
        ↓  ← CRITICAL: without this, drainage analysis is meaningless
Step 5: Flow direction (D8 algorithm)
        ↓
Step 6: Flow accumulation → stream delineation
        ↓
Step 7: Export as GeoTIFF (DTM + drainage network)
```

## Why each step matters for drainage
| Step | Why it is non-negotiable |
|---|---|
| Ground-only points | Buildings/trees create fake ridges that completely block water flow |
| IDW interpolation | Sparse points → continuous surface water can physically flow across |
| Sink-filling | Even 1-pixel depressions trap water and break flow accumulation |
| D8 flow direction | Determines where each cell drains — the basis of all drainage calculations |
| Flow accumulation | Identifies natural drainage paths (high accumulation = stream channel) |
| GeoTIFF + CRS | Allows import into QGIS, ArcGIS, SWMM, EPA-Net for actual design |

## Outputs
- `dtm_clean.tif`           — Cleaned DTM raster, metric elevations, projected CRS
- `dtm_conditioned.tif`     — Hydrologically conditioned (sinks filled)
- `flow_direction.tif`      — D8 flow direction grid
- `flow_accumulation.tif`   — Upstream contributing area per cell
- `drainage_streams.tif`    — Binary stream network raster
- `drainage_streams.gpkg`   — Vector drainage network (import to QGIS/ArcGIS)
- `ground_points.las`       — Classified ground-only LAS file (open in CloudCompare)


In [ ]:
# ─── Cell 1: Install dependencies ────────────────────────────────────────────
# pysheds  — hydrological analysis (sink-filling, flow direction, accumulation)
# rasterio — GeoTIFF read/write with CRS support
# fiona    — vector drainage network export (GeoPackage)
# shapely  — geometry for vectorisation
!pip install --quiet pysheds rasterio fiona shapely scipy numpy tqdm matplotlib laspy

In [ ]:
# ─── Cell 2: Configuration ────────────────────────────────────────────────────
from pathlib import Path
import torch
import numpy as np
import os

# ── INPUTS — set these paths ──────────────────────────────────────────────────
TRAINED_MODEL_PATH = '/kaggle/working/logs/best_model.pth'
DATASET_ROOT       = '/kaggle/input/datasets/jaideepchouhan/point-cloud-data-of-10-indian-villages/Training'

# ── OUTPUTS ───────────────────────────────────────────────────────────────────
OUTPUT_DIR         = '/kaggle/working/dtm_outputs'

# ── Coordinate Reference System ───────────────────────────────────────────────
# CRITICAL: Set this to the actual CRS of your data.
# Indian village LiDAR is typically in one of these:
#   EPSG:32643  — WGS84 / UTM zone 43N  (most of Rajasthan, Gujarat)
#   EPSG:32644  — WGS84 / UTM zone 44N  (eastern Rajasthan, MP, UP)
#   EPSG:32645  — WGS84 / UTM zone 45N  (eastern UP, Bihar)
#   EPSG:7760   — WGS84 / India NSF zone (national grid)
# If you do not know, check with: gdalinfo your_file.las | grep -i crs
DATA_CRS_EPSG = 32643    # ← CHANGE THIS if needed

# ── DTM raster resolution ─────────────────────────────────────────────────────
# 0.5 m = very high detail (large file, slow)
# 1.0 m = good for drainage design in Indian villages  ← RECOMMENDED
# 2.0 m = faster processing, coarser drainage paths
RASTER_RESOLUTION_M = 1.0

# ── Stream delineation threshold ─────────────────────────────────────────────
# A cell is considered a stream if >= STREAM_THRESHOLD upstream cells drain to it.
# Lower = more streams (denser network).  Higher = only main channels.
# For 1 m resolution in village terrain:  500–2000 is typical
STREAM_THRESHOLD = 1000

# ── Inference settings ────────────────────────────────────────────────────────
# For DTM we run on FULL point cloud (not sub-sampled to 4096)
INFERENCE_CHUNK   = 8192    # process this many points at once (GPU memory)
INFERENCE_SPLIT   = 'val'   # 'train' | 'val' | 'both' — which tiles to process
INFERENCE_WORKERS = 4
CONFIDENCE_THRESH = 0.55    # ground class probability threshold (from threshold sweep)

# ─────────────────────────────────────────────────────────────────────────────
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print(f'Output : {OUTPUT_DIR}')
print(f'CRS    : EPSG:{DATA_CRS_EPSG}')
print(f'Res    : {RASTER_RESOLUTION_M} m')
print(f'Stream threshold: {STREAM_THRESHOLD} cells')

In [ ]:
# ─── Cell 3: Load trained model ───────────────────────────────────────────────
#
# The model architecture must exactly match the training notebook.
# Copy the full DTMPointNet2 class definition here.
# ─────────────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F


def farthest_point_sample(xyz, n_sample):
    B, N, _ = xyz.shape
    dev  = xyz.device
    cent = torch.zeros(B, n_sample, dtype=torch.long,  device=dev)
    dist = torch.full((B, N), 1e10, dtype=torch.float, device=dev)
    far  = torch.randint(0, N, (B,), dtype=torch.long, device=dev)
    bidx = torch.arange(B, dtype=torch.long, device=dev)
    for i in range(n_sample):
        cent[:, i] = far
        c   = xyz[bidx, far, :].unsqueeze(1)
        d   = ((xyz - c) ** 2).sum(dim=-1)
        dist = torch.where(d < dist, d, dist)
        far  = dist.argmax(dim=-1)
    return cent


def index_points(pts, idx):
    B    = pts.shape[0]
    shp  = [B] + [1] * (idx.dim() - 1)
    bidx = torch.arange(B, device=pts.device).view(shp).expand_as(idx)
    return pts[bidx, idx]


def ball_query_topk(new_xyz, xyz, radius, n_samp):
    dist    = torch.cdist(new_xyz, xyz)
    masked  = torch.where(dist <= radius, dist, dist.new_full((), 1e10))
    return masked.topk(n_samp, dim=-1, largest=False)[1]


class SetAbstractionMSG(nn.Module):
    def __init__(self, n_ctr, radii, samples, in_ch, mlp_specs):
        super().__init__()
        self.n_ctr, self.radii, self.samples = n_ctr, radii, samples
        self.mlps = nn.ModuleList()
        for mlp_dims in mlp_specs:
            layers, last = [], in_ch + 3
            for d in mlp_dims:
                layers += [nn.Conv2d(last, d, 1, bias=False), nn.BatchNorm2d(d), nn.ReLU(inplace=True)]
                last = d
            self.mlps.append(nn.Sequential(*layers))

    def forward(self, xyz, points):
        fps_idx = farthest_point_sample(xyz, self.n_ctr)
        new_xyz = index_points(xyz, fps_idx)
        dist    = torch.cdist(new_xyz, xyz)
        outs    = []
        for r, k, mlp in zip(self.radii, self.samples, self.mlps):
            masked  = torch.where(dist <= r, dist, dist.new_full((), 1e10))
            idx     = masked.topk(k, dim=-1, largest=False)[1]
            grp_xyz = index_points(xyz, idx) - new_xyz.unsqueeze(2)
            grp_pts = (torch.cat([grp_xyz, index_points(points, idx)], dim=-1)
                       if points is not None else grp_xyz)
            grp_pts = grp_pts.permute(0, 3, 2, 1)
            feat    = mlp(grp_pts).max(dim=2)[0].permute(0, 2, 1)
            outs.append(feat)
        return new_xyz, torch.cat(outs, dim=-1)


class SetAbstraction(nn.Module):
    def __init__(self, n_ctr, radius, n_samp, in_ch, mlp_dims):
        super().__init__()
        self.n_ctr, self.radius, self.n_samp = n_ctr, radius, n_samp
        layers, last = [], in_ch + 3
        for d in mlp_dims:
            layers += [nn.Conv2d(last, d, 1, bias=False), nn.BatchNorm2d(d), nn.ReLU(inplace=True)]
            last = d
        self.mlp = nn.Sequential(*layers)

    def forward(self, xyz, points):
        fps_idx = farthest_point_sample(xyz, self.n_ctr)
        new_xyz = index_points(xyz, fps_idx)
        idx     = ball_query_topk(new_xyz, xyz, self.radius, self.n_samp)
        grp_xyz = index_points(xyz, idx) - new_xyz.unsqueeze(2)
        grp_pts = (torch.cat([grp_xyz, index_points(points, idx)], dim=-1)
                   if points is not None else grp_xyz)
        feat    = self.mlp(grp_pts.permute(0, 3, 2, 1)).max(dim=2)[0].permute(0, 2, 1)
        return new_xyz, feat


class FeaturePropagation(nn.Module):
    def __init__(self, in_ch, mlp_dims):
        super().__init__()
        layers, last = [], in_ch
        for d in mlp_dims:
            layers += [nn.Conv1d(last, d, 1, bias=False), nn.BatchNorm1d(d), nn.ReLU(inplace=True)]
            last = d
        self.mlp = nn.Sequential(*layers)

    def forward(self, xyz1, xyz2, pts1, pts2):
        dists, idx = torch.cdist(xyz1, xyz2).topk(3, dim=-1, largest=False)
        dists = torch.clamp(dists, min=1e-10)
        w     = 1.0 / dists
        w     = w / w.sum(dim=-1, keepdim=True)
        interp = (index_points(pts2, idx) * w.unsqueeze(-1)).sum(dim=2)
        feat   = torch.cat([pts1, interp], dim=-1) if pts1 is not None else interp
        return self.mlp(feat.permute(0, 2, 1)).permute(0, 2, 1)


class DTMPointNet2(nn.Module):
    IN_FEAT_CH = 4
    def __init__(self, num_classes=2):
        super().__init__()
        IC = self.IN_FEAT_CH
        self.sa1 = SetAbstractionMSG(512, [0.5, 1.5], [32, 64], IC,  [[32,64],[64,128]])
        self.sa2 = SetAbstractionMSG(128, [3.0, 6.0], [64,128], 192, [[128,128],[128,256]])
        self.sa3 = SetAbstraction(32, 12.0, 128, 384, [256, 512])
        self.fp3 = FeaturePropagation(384 + 512, [512, 256])
        self.fp2 = FeaturePropagation(192 + 256, [256, 128])
        self.fp1 = FeaturePropagation(IC  + 128, [128, 128])
        self.head = nn.Sequential(
            nn.Conv1d(128, 128, 1, bias=False), nn.BatchNorm1d(128), nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Conv1d(128, 64, 1, bias=False), nn.BatchNorm1d(64), nn.ReLU(inplace=True),
            nn.Conv1d(64, num_classes, 1),
        )

    def forward(self, x):
        xyz    = x[:, :, :3].contiguous()
        l0_pts = x[:, :, 3:].contiguous()
        l1_xyz, l1_pts = self.sa1(xyz,    l0_pts)
        l2_xyz, l2_pts = self.sa2(l1_xyz, l1_pts)
        l3_xyz, l3_pts = self.sa3(l2_xyz, l2_pts)
        l2_pts = self.fp3(l2_xyz, l3_xyz, l2_pts, l3_pts)
        l1_pts = self.fp2(l1_xyz, l2_xyz, l1_pts, l2_pts)
        l0_pts = self.fp1(xyz,    l1_xyz, l0_pts, l1_pts)
        out    = self.head(l0_pts.permute(0, 2, 1))
        return out.permute(0, 2, 1)


# ── Load weights ──────────────────────────────────────────────────────────────
model = DTMPointNet2(num_classes=2).to(device)
state = torch.load(TRAINED_MODEL_PATH, map_location=device)
# Handle both raw state_dict and wrapped checkpoint
if 'model_state_dict' in state:
    state = state['model_state_dict']
model.load_state_dict(state)
model.eval()
print(f'✅ Model loaded from {TRAINED_MODEL_PATH}')
print(f'   Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ─── Cell 4: Geospatial feature computation (same as training) ─────────────────
import numpy as np

def compute_geospatial_features(xyz: np.ndarray) -> np.ndarray:
    """
    Compute [delta_z, roughness, slope, density] for each point.
    Must be IDENTICAL to the function used during training.
    """
    xyz  = xyz.astype(np.float64)
    x_min, y_min = xyz[:, 0].min(), xyz[:, 1].min()
    x_rng = xyz[:, 0].max() - x_min
    y_rng = xyz[:, 1].max() - y_min
    GW = int(np.clip(x_rng / 2.0, 16, 64))
    GH = int(np.clip(y_rng / 2.0, 16, 64))
    xi = np.clip(((xyz[:, 0] - x_min) / (x_rng + 1e-6) * GW).astype(np.int32), 0, GW-1)
    yi = np.clip(((xyz[:, 1] - y_min) / (y_rng + 1e-6) * GH).astype(np.int32), 0, GH-1)
    ci = yi * GW + xi
    NC = GH * GW
    z  = xyz[:, 2]
    c_min = np.full(NC, np.inf);  c_sum = np.zeros(NC); c_sq = np.zeros(NC); c_cnt = np.zeros(NC)
    np.minimum.at(c_min, ci, z);  np.add.at(c_sum, ci, z)
    np.add.at(c_sq, ci, z*z);     np.add.at(c_cnt, ci, 1.0)
    empty = c_cnt == 0;  z_gm = z.mean()
    c_cnt[empty]=1.;  c_min[empty]=z_gm;  c_sum[empty]=z_gm;  c_sq[empty]=z_gm**2
    c_mean = c_sum / c_cnt
    c_std  = np.sqrt(np.maximum(c_sq / c_cnt - c_mean**2, 0.))
    delta_z   = np.clip(z - c_min[ci], 0., None).astype(np.float32)
    roughness = c_std[ci].astype(np.float32)
    dz_dy, dz_dx = np.gradient(c_min.reshape(GH, GW))
    slope  = np.sqrt(dz_dx**2 + dz_dy**2).ravel()[ci].astype(np.float32)
    density = (c_cnt[ci] / (c_cnt.max() + 1e-6)).astype(np.float32)
    feat = np.stack([delta_z, roughness, slope, density], axis=1)
    for i in range(4):
        mu, sigma = feat[:, i].mean(), feat[:, i].std() + 1e-6
        feat[:, i] = (feat[:, i] - mu) / sigma
    return feat


print('✅ Feature function ready')

In [ ]:
# ─── Cell 5: Full-resolution inference — extract ground points ─────────────────
#
# KEY DIFFERENCE FROM TRAINING:
# We run inference on the COMPLETE point cloud of each tile, not a 4096-point sample.
# This ensures we recover ALL ground points, not just a fraction.
# To fit GPU memory, we chunk the point cloud into INFERENCE_CHUNK-sized blocks.
# ─────────────────────────────────────────────────────────────────────────────
import torch
import torch.nn.functional as F
import numpy as np
from pathlib import Path
from tqdm    import tqdm
import time


def infer_tile_ground_points(
    model        : torch.nn.Module,
    xyz_full     : np.ndarray,      # (N_full, 3) — FULL tile, not sub-sampled
    threshold    : float = 0.55,
    chunk_size   : int   = 8192,
    device       = None,
) -> np.ndarray:
    """
    Run ground classification on the full tile (chunked for GPU memory).

    Strategy: slide a window of `chunk_size` points across the tile.
    Each chunk is a spatially coherent block (sorted by XY) so that
    the local neighbourhood features remain meaningful.

    Returns
    -------
    ground_xyz : (M, 3)  — XYZ of points classified as ground
    """
    N = len(xyz_full)
    if N < 64:
        return np.empty((0, 3), dtype=np.float32)

    # Sort points spatially (x-major) so chunks are spatially coherent
    sort_idx = np.lexsort((xyz_full[:, 1], xyz_full[:, 0]))
    xyz_sorted = xyz_full[sort_idx].astype(np.float32)

    # Compute geospatial features on the FULL tile (not per-chunk)
    # This is correct: ΔZ and slope need the full tile context
    feat_extra = compute_geospatial_features(xyz_sorted)  # (N, 4)

    ground_mask = np.zeros(N, dtype=bool)

    model.eval()
    with torch.no_grad():
        for start in range(0, N, chunk_size):
            end  = min(start + chunk_size, N)
            cxyz = xyz_sorted[start:end]         # (C, 3)
            cfeat = feat_extra[start:end]         # (C, 4)
            C    = cxyz.shape[0]

            # Pad to at least 64 points (model minimum)
            if C < 64:
                pad_n  = 64 - C
                pad_idx = np.random.choice(C, pad_n, replace=True)
                cxyz   = np.concatenate([cxyz,  cxyz[pad_idx]], axis=0)
                cfeat  = np.concatenate([cfeat, cfeat[pad_idx]], axis=0)

            # Normalise XY to zero-mean (same as training)
            cxyz[:, 0] -= cxyz[:, 0].mean()
            cxyz[:, 1] -= cxyz[:, 1].mean()

            feat_tensor = np.concatenate([cxyz, cfeat], axis=1)  # (C, 7)
            feat_tensor = torch.from_numpy(feat_tensor).unsqueeze(0).to(device)  # (1, C, 7)

            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                logits = model(feat_tensor)           # (1, C, 2)
            probs  = F.softmax(logits, dim=-1)[0, :C, 1].float().cpu().numpy()  # ground prob

            ground_mask[start:end] = probs[:C] >= threshold

    ground_xyz = xyz_sorted[ground_mask]
    return ground_xyz


def run_inference_all_tiles(
    model        : torch.nn.Module,
    dataset_root : str,
    split        : str   = 'val',
    threshold    : float = 0.55,
    chunk_size   : int   = 8192,
    device       = None,
    max_tiles    : int   = None,   # None = all tiles
) -> tuple:
    """
    Run inference across all tiles and collect ground points.

    Returns
    -------
    all_ground_xyz : (M_total, 3) — all ground points from all tiles
    stats          : dict          — per-tile statistics
    """
    split_root  = Path(dataset_root) / split
    tile_dirs   = sorted([p for p in split_root.glob('tile_*')
                          if (p / 'points.npy').exists()])

    if not tile_dirs:
        raise FileNotFoundError(f'No tiles found in {split_root}')

    if max_tiles is not None:
        tile_dirs = tile_dirs[:max_tiles]

    print(f'Processing {len(tile_dirs)} tiles from [{split}]')
    print(f'Threshold     : {threshold}')
    print(f'Chunk size    : {chunk_size} pts')

    all_chunks  = []
    stats       = []
    t0          = time.time()

    for tile_dir in tqdm(tile_dirs, desc='Inference', ncols=80):
        try:
            xyz = np.load(tile_dir / 'points.npy').astype(np.float32)[:, :3]
            n_total = len(xyz)

            ground_xyz = infer_tile_ground_points(
                model, xyz, threshold=threshold,
                chunk_size=chunk_size, device=device
            )
            n_ground = len(ground_xyz)

            if n_ground > 0:
                all_chunks.append(ground_xyz)

            stats.append({
                'tile'      : tile_dir.name,
                'n_total'   : n_total,
                'n_ground'  : n_ground,
                'ratio'     : n_ground / (n_total + 1e-6),
            })

        except Exception as e:
            print(f'  ⚠️  {tile_dir.name}: {e}')

    all_ground = np.concatenate(all_chunks, axis=0) if all_chunks else np.empty((0, 3), np.float32)

    elapsed = time.time() - t0
    ratios  = [s['ratio'] for s in stats]
    print(f'\n✅ Inference complete in {elapsed:.0f}s')
    print(f'   Total ground points : {len(all_ground):,}')
    print(f'   Avg ground ratio    : {np.mean(ratios):.1%}')
    print(f'   Min / Max ratio     : {min(ratios):.1%} / {max(ratios):.1%}')

    return all_ground, stats


# ── Run ───────────────────────────────────────────────────────────────────────
all_ground_xyz, inference_stats = run_inference_all_tiles(
    model        = model,
    dataset_root = DATASET_ROOT,
    split        = INFERENCE_SPLIT,
    threshold    = CONFIDENCE_THRESH,
    chunk_size   = INFERENCE_CHUNK,
    device       = device,
)

print(f'\nGround point cloud extent:')
print(f'  X: [{all_ground_xyz[:,0].min():.2f}, {all_ground_xyz[:,0].max():.2f}] m')
print(f'  Y: [{all_ground_xyz[:,1].min():.2f}, {all_ground_xyz[:,1].max():.2f}] m')
print(f'  Z: [{all_ground_xyz[:,2].min():.2f}, {all_ground_xyz[:,2].max():.2f}] m')

In [ ]:
# ─── Cell 6: IDW Interpolation → clean DTM raster ────────────────────────────
#
# Inverse Distance Weighting (IDW) interpolation:
#   - For each raster cell, find the K nearest ground points
#   - Weight their Z values by 1/distance^power
#   - This produces a smooth, continuous surface from sparse ground points
#
# For drainage design, IDW with power=2, K=8 is standard.
# Kriging would be more accurate but is ~50× slower — impractical here.
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
from scipy.spatial import cKDTree
import rasterio
from rasterio.transform import from_origin
from rasterio.crs import CRS
from pathlib import Path
import time


def idw_interpolate(
    ground_xyz    : np.ndarray,
    resolution_m  : float = 1.0,
    k_neighbors   : int   = 8,
    power         : float = 2.0,
    max_search_m  : float = 10.0,   # ignore ground points farther than this
    nodata_val    : float = -9999.0,
) -> tuple:
    """
    Interpolate ground points to a regular raster grid using IDW.

    Returns
    -------
    dtm_grid    : (rows, cols) float32 — elevation values
    transform   : rasterio Affine      — geotransform
    x_origin    : float                — left edge easting
    y_origin    : float                — top edge northing
    """
    print('Building KD-tree from ground points...')
    t0 = time.time()

    x, y, z = ground_xyz[:, 0], ground_xyz[:, 1], ground_xyz[:, 2]

    # Grid extent with half-cell buffer
    x_min, x_max = x.min() - resolution_m, x.max() + resolution_m
    y_min, y_max = y.min() - resolution_m, y.max() + resolution_m

    cols = int(np.ceil((x_max - x_min) / resolution_m))
    rows = int(np.ceil((y_max - y_min) / resolution_m))
    print(f'  Grid size: {rows} rows × {cols} cols  ({rows*cols/1e6:.2f}M cells)')

    # Rasterio convention: origin is top-left, Y decreases downward
    x_origin = x_min
    y_origin = y_max
    transform = from_origin(x_origin, y_origin, resolution_m, resolution_m)

    # Build KD-tree on XY only
    tree = cKDTree(np.column_stack([x, y]))
    print(f'  KD-tree built in {time.time()-t0:.1f}s')

    # Build query points (cell centres)
    col_idx = np.arange(cols)
    row_idx = np.arange(rows)
    gx = x_origin + (col_idx + 0.5) * resolution_m      # (cols,)
    gy = y_origin - (row_idx + 0.5) * resolution_m      # (rows,)
    GX, GY = np.meshgrid(gx, gy)                         # (rows, cols)
    query_pts = np.column_stack([GX.ravel(), GY.ravel()])

    print(f'  Running IDW query (k={k_neighbors})...')
    t1 = time.time()

    # Batch query for all cells
    dists, idxs = tree.query(query_pts, k=k_neighbors, workers=-1)
    print(f'  Query done in {time.time()-t1:.1f}s')

    # IDW weights
    dists  = np.maximum(dists, 1e-6)                     # avoid div/0
    within = dists[:, 0] <= max_search_m                 # mask far cells
    weights = 1.0 / (dists ** power)                     # (M, k)
    w_sum   = weights.sum(axis=1, keepdims=True)
    z_interp = (weights * z[idxs]).sum(axis=1) / w_sum.ravel()  # (M,)

    dtm_flat = np.where(within, z_interp, nodata_val).astype(np.float32)
    dtm_grid = dtm_flat.reshape(rows, cols)

    valid_cells = (dtm_grid != nodata_val).sum()
    print(f'  IDW complete in {time.time()-t0:.1f}s')
    print(f'  Valid cells   : {valid_cells:,} / {rows*cols:,} ({100*valid_cells/(rows*cols):.1f}%)')
    print(f'  Z range       : {dtm_grid[dtm_grid != nodata_val].min():.2f} – '
          f'{dtm_grid[dtm_grid != nodata_val].max():.2f} m')

    return dtm_grid, transform, x_origin, y_origin


def save_geotiff(
    grid       : np.ndarray,
    transform,
    crs_epsg   : int,
    out_path   : str,
    nodata_val : float = -9999.0,
    layer_name : str   = 'DTM',
) -> None:
    """Save a 2D numpy array as a single-band GeoTIFF."""
    with rasterio.open(
        out_path,
        'w',
        driver     = 'GTiff',
        height     = grid.shape[0],
        width      = grid.shape[1],
        count      = 1,
        dtype      = grid.dtype,
        crs        = CRS.from_epsg(crs_epsg),
        transform  = transform,
        nodata     = nodata_val,
        compress   = 'lzw',
        predictor  = 2,
        tiled      = True,
        blockxsize = 256,
        blockysize = 256,
    ) as dst:
        dst.write(grid, 1)
        dst.update_tags(layer=layer_name, source='DTM_PointNet2_AI')
    size_mb = Path(out_path).stat().st_size / 1e6
    print(f'  Saved {layer_name}: {out_path}  ({size_mb:.1f} MB)')


# ── Run IDW interpolation ─────────────────────────────────────────────────────
print('Running IDW interpolation...')
dtm_grid, dtm_transform, x_orig, y_orig = idw_interpolate(
    all_ground_xyz,
    resolution_m = RASTER_RESOLUTION_M,
    k_neighbors  = 8,
    power        = 2.0,
    max_search_m = 10.0,
)

# ── Save clean DTM ────────────────────────────────────────────────────────────
dtm_path = f'{OUTPUT_DIR}/dtm_clean.tif'
save_geotiff(dtm_grid, dtm_transform, DATA_CRS_EPSG, dtm_path, layer_name='DTM_Clean')

# ── Quick visualisation ───────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
valid = dtm_grid != -9999.0
vmin  = np.percentile(dtm_grid[valid], 2)
vmax  = np.percentile(dtm_grid[valid], 98)

im0 = axes[0].imshow(dtm_grid, cmap='terrain', vmin=vmin, vmax=vmax)
axes[0].set_title('Clean DTM (AI ground classification + IDW)')
plt.colorbar(im0, ax=axes[0], label='Elevation (m)')

# Hillshade for visual inspection
from matplotlib.colors import LightSource
ls   = LightSource(azdeg=315, altdeg=45)
dtm_vis = np.where(valid, dtm_grid, np.nan)
hs   = ls.hillshade(dtm_vis, vert_exag=3, dx=RASTER_RESOLUTION_M, dy=RASTER_RESOLUTION_M)
axes[1].imshow(hs, cmap='gray')
axes[1].set_title('Hillshade (verify buildings/trees are absent)')

plt.suptitle('DTM Quality Check — Ground Points Only', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/dtm_visual_check.png', dpi=150)
plt.show()
print('✅ DTM visualisation saved')

In [ ]:
# ─── Cell 7: Hydrological conditioning — CRITICAL for drainage ───────────────
#
# Without this step, EVERY micro-depression in the DTM (including LiDAR noise)
# creates a "sink" where water pools and cannot flow further downstream.
# This would make flow accumulation and stream delineation completely wrong.
#
# We use pysheds which implements the Priority-Flood algorithm:
# Wang & Liu (2006) — An efficient method for identifying and filling surface depressions
# in digital elevation models for hydrologic analysis and modelling.
#
# Priority-Flood fills every depression to the level of its lowest rim,
# ensuring a continuous, connected flow path from every cell to the outlet.
# ─────────────────────────────────────────────────────────────────────────────
from pysheds.grid import Grid
import numpy as np
from pathlib import Path
import rasterio


print('Loading DTM into pysheds...')
grid   = Grid.from_raster(dtm_path)
dem    = grid.read_raster(dtm_path)

print('Conditioning DTM (filling sinks + resolving flats)...')

# Step 1: Fill pits (isolated single-cell depressions — common LiDAR artefacts)
pit_filled  = grid.fill_pits(dem)

# Step 2: Fill depressions (larger enclosed basins)
# This is the Priority-Flood step
flooded     = grid.fill_depressions(pit_filled)

# Step 3: Resolve flat areas (cells at exactly the same elevation)
# Without this, D8 cannot determine flow direction on flat plateaus
conditioned = grid.resolve_flats(flooded)

print('✅ Hydrological conditioning complete')

# ── Save conditioned DTM ──────────────────────────────────────────────────────
dtm_cond_path = f'{OUTPUT_DIR}/dtm_conditioned.tif'
grid.to_raster(conditioned, dtm_cond_path)
print(f'   Saved: {dtm_cond_path}')

# ── Visualise what changed ────────────────────────────────────────────────────
import matplotlib.pyplot as plt

dtm_arr   = np.array(dem)
cond_arr  = np.array(conditioned)
diff      = cond_arr - dtm_arr
n_changed = (np.abs(diff) > 0.001).sum()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].imshow(dtm_arr,  cmap='terrain'); axes[0].set_title('Raw DTM')
axes[1].imshow(cond_arr, cmap='terrain'); axes[1].set_title('Conditioned DTM')
im = axes[2].imshow(diff, cmap='RdBu', vmin=-1, vmax=1)
axes[2].set_title(f'Difference (filled cells: {n_changed:,})')
plt.colorbar(im, ax=axes[2], label='Elevation change (m)')
plt.suptitle('Hydrological Conditioning Effect', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/conditioning_diff.png', dpi=150)
plt.show()
print(f'   Cells modified: {n_changed:,}')

In [ ]:
# ─── Cell 8: Flow direction (D8) + Flow accumulation ─────────────────────────
#
# D8 algorithm (O'Callaghan & Mark 1984):
#   Each raster cell drains to the steepest of its 8 neighbours.
#   Encoded as: 1=E, 2=SE, 4=S, 8=SW, 16=W, 32=NW, 64=N, 128=NE
#
# Flow accumulation:
#   For each cell, count the number of upstream cells that drain into it.
#   High accumulation values identify channels and stream paths.
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm


print('Computing D8 flow direction...')
fdir = grid.flowdir(conditioned)

print('Computing flow accumulation...')
acc  = grid.accumulation(fdir)

fdir_arr = np.array(fdir)
acc_arr  = np.array(acc)

print(f'✅ Flow analysis complete')
print(f'   Max accumulation: {acc_arr.max():,.0f} cells')
print(f'   Cells with acc > {STREAM_THRESHOLD}: {(acc_arr > STREAM_THRESHOLD).sum():,}')

# ── Save ──────────────────────────────────────────────────────────────────────
fdir_path = f'{OUTPUT_DIR}/flow_direction.tif'
acc_path  = f'{OUTPUT_DIR}/flow_accumulation.tif'
grid.to_raster(fdir, fdir_path)
grid.to_raster(acc,  acc_path)
print(f'   Saved: {fdir_path}')
print(f'   Saved: {acc_path}')

# ── Visualise ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(fdir_arr, cmap='tab10')
axes[0].set_title('D8 Flow direction (8 directions)')

# Log scale for accumulation — reveals stream hierarchy clearly
acc_vis = np.where(acc_arr > 0, acc_arr, np.nan)
im = axes[1].imshow(acc_vis, cmap='Blues', norm=LogNorm(vmin=1, vmax=acc_arr.max()))
plt.colorbar(im, ax=axes[1], label='Upstream cells (log scale)')
axes[1].set_title('Flow accumulation')

plt.suptitle('Hydrology Analysis', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/flow_analysis.png', dpi=150)
plt.show()

In [ ]:
# ─── Cell 9: Stream delineation + vector drainage network ─────────────────────
#
# A cell is a stream if it accumulates >= STREAM_THRESHOLD upstream cells.
# We also vectorise the network as linestrings for GIS import.
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.features import shapes
import fiona
from fiona.crs import from_epsg
from shapely.geometry import shape, mapping
from pathlib import Path


# ── Extract stream raster ─────────────────────────────────────────────────────
print(f'Extracting streams (threshold = {STREAM_THRESHOLD} cells)...')
branches = grid.extract_river_network(fdir, acc > STREAM_THRESHOLD)

# Binary stream mask
stream_mask = (acc_arr >= STREAM_THRESHOLD).astype(np.uint8)

stream_path = f'{OUTPUT_DIR}/drainage_streams.tif'
with rasterio.open(
    stream_path, 'w', driver='GTiff',
    height=stream_mask.shape[0], width=stream_mask.shape[1],
    count=1, dtype='uint8',
    crs=rasterio.crs.CRS.from_epsg(DATA_CRS_EPSG),
    transform=dtm_transform,
    compress='lzw',
) as dst:
    dst.write(stream_mask, 1)

print(f'   Stream cells   : {stream_mask.sum():,}')
print(f'   Approx length  : {stream_mask.sum() * RASTER_RESOLUTION_M / 1000:.1f} km')
print(f'   Saved raster   : {stream_path}')


# ── Vectorise drainage network (export as GeoPackage) ────────────────────────
vector_path = f'{OUTPUT_DIR}/drainage_streams.gpkg'

schema = {
    'geometry'  : 'LineString',
    'properties': {'stream_order': 'int', 'acc_cells': 'float'},
}

n_features = 0
with fiona.open(vector_path, 'w', driver='GPKG',
                crs=from_epsg(DATA_CRS_EPSG), schema=schema) as f:
    for branch in branches['features']:
        geom  = branch['geometry']
        props = branch.get('properties', {})
        f.write({
            'geometry'  : geom,
            'properties': {
                'stream_order': int(props.get('order', 1)),
                'acc_cells'   : float(props.get('accumulation', 0)),
            },
        })
        n_features += 1

print(f'   Vector segments: {n_features:,}')
print(f'   Saved vector   : {vector_path}')


# ── Final visualisation — DTM + drainage overlay ──────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))

cond_arr = np.array(conditioned)
valid    = cond_arr > -9999
vmin     = np.percentile(cond_arr[valid], 2)
vmax     = np.percentile(cond_arr[valid], 98)

ax.imshow(cond_arr, cmap='terrain', vmin=vmin, vmax=vmax, alpha=0.85)

# Overlay streams in blue, thicker for higher order channels
stream_overlay = np.ma.masked_where(stream_mask == 0, stream_mask)
ax.imshow(stream_overlay, cmap='Blues', alpha=0.9, vmin=0, vmax=1)

ax.set_title(
    f'AI-Generated DTM + Drainage Network\n'
    f'Resolution: {RASTER_RESOLUTION_M}m  |  '
    f'Stream threshold: {STREAM_THRESHOLD} cells  |  '
    f'Network length: {stream_mask.sum() * RASTER_RESOLUTION_M / 1000:.1f} km',
    fontsize=11
)
ax.set_xlabel('Column (pixel)')
ax.set_ylabel('Row (pixel)')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/dtm_with_drainage.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'   Saved overview : {OUTPUT_DIR}/dtm_with_drainage.png')

In [ ]:
# ─── Cell 10: Optional — export ground-only LAS file ─────────────────────────
#
# LAS format can be opened in CloudCompare, LiDAR360, or PDAL.
# Useful for visual inspection of the ground classification quality.
# ─────────────────────────────────────────────────────────────────────────────
try:
    import laspy
    import numpy as np

    print('Exporting ground-only LAS file...')

    header = laspy.LasHeader(version='1.4', point_format=0)
    header.offsets  = all_ground_xyz.min(axis=0)
    header.scales   = np.array([0.001, 0.001, 0.001])   # 1 mm precision

    las = laspy.LasData(header=header)
    las.x = all_ground_xyz[:, 0]
    las.y = all_ground_xyz[:, 1]
    las.z = all_ground_xyz[:, 2]
    las.classification = np.full(len(all_ground_xyz), 2, dtype=np.uint8)  # LAS class 2 = ground

    las_path = f'{OUTPUT_DIR}/ground_points.las'
    las.write(las_path)

    from pathlib import Path
    size_mb = Path(las_path).stat().st_size / 1e6
    print(f'✅ LAS saved: {las_path}  ({size_mb:.1f} MB)')
    print(f'   Points: {len(all_ground_xyz):,}')
    print(f'   Classification: 2 (ground) for all points')
    print('   Open in: CloudCompare, QGIS, LAStools, PDAL')

except ImportError:
    print('ℹ️  laspy not available — skipping LAS export (run: pip install laspy)')
except Exception as e:
    print(f'⚠️  LAS export error: {e}')

In [ ]:
# ─── Cell 11: Final summary and output collection ─────────────────────────────
import zipfile
from pathlib import Path
import json

outputs = [
    f'{OUTPUT_DIR}/dtm_clean.tif',
    f'{OUTPUT_DIR}/dtm_conditioned.tif',
    f'{OUTPUT_DIR}/flow_direction.tif',
    f'{OUTPUT_DIR}/flow_accumulation.tif',
    f'{OUTPUT_DIR}/drainage_streams.tif',
    f'{OUTPUT_DIR}/drainage_streams.gpkg',
    f'{OUTPUT_DIR}/dtm_with_drainage.png',
    f'{OUTPUT_DIR}/dtm_visual_check.png',
    f'{OUTPUT_DIR}/conditioning_diff.png',
    f'{OUTPUT_DIR}/flow_analysis.png',
]

# Save pipeline metadata
meta = {
    'model_path'         : TRAINED_MODEL_PATH,
    'dataset_root'       : DATASET_ROOT,
    'inference_split'    : INFERENCE_SPLIT,
    'confidence_threshold': CONFIDENCE_THRESH,
    'crs_epsg'           : DATA_CRS_EPSG,
    'resolution_m'       : RASTER_RESOLUTION_M,
    'stream_threshold'   : STREAM_THRESHOLD,
    'total_ground_pts'   : int(len(all_ground_xyz)),
    'stream_length_km'   : round(float(stream_mask.sum() * RASTER_RESOLUTION_M / 1000), 2),
}
meta_path = f'{OUTPUT_DIR}/pipeline_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
outputs.append(meta_path)

# Bundle everything
bundle_path = '/kaggle/working/dtm_drainage_bundle.zip'
with zipfile.ZipFile(bundle_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in outputs:
        if Path(p).exists():
            z.write(p, arcname=Path(p).name)

print('\n' + '='*62)
print('  DTM GENERATION PIPELINE — COMPLETE')
print('='*62)
print(f'  Ground points extracted : {len(all_ground_xyz):,}')
print(f'  DTM resolution          : {RASTER_RESOLUTION_M} m')
print(f'  Drainage network length : {stream_mask.sum() * RASTER_RESOLUTION_M / 1000:.1f} km')
print(f'  CRS                     : EPSG:{DATA_CRS_EPSG}')
print()
print('  Output files:')
for p in outputs:
    ok   = Path(p).exists()
    stat = '✅' if ok else '❌'
    size = f'  {Path(p).stat().st_size/1e6:.1f}MB' if ok else ''
    print(f'  {stat} {Path(p).name}{size}')
print()
print(f'  Bundle: {bundle_path}')
print('='*62)
print()
print('  Next steps in QGIS / ArcGIS:')
print('  1. Load dtm_conditioned.tif  → base terrain surface')
print('  2. Load drainage_streams.gpkg → drainage network vector')
print('  3. Load flow_accumulation.tif → identify main channels')
print('  4. Design drainage infrastructure on top of this DTM')
print('  5. Run SWMM / EPA-SWMM for hydraulic routing simulation')